In [ ]:
import os

INPUT_BASE = '/kaggle/input/datasets/msebastiaanh/5gcn-2905/HAABSA7GCN'

assert os.path.isdir(f'{INPUT_BASE}/src'), f"src/ not found under {INPUT_BASE}"
assert os.path.isdir(f'{INPUT_BASE}/data'), f"data/ not found under {INPUT_BASE}"
print('src/:', sorted(os.listdir(f'{INPUT_BASE}/src')))
print('data/:', sorted(os.listdir(f'{INPUT_BASE}/data')))

In [ ]:
import shutil, os

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/data', exist_ok=True)

# Copy all data files (including .const, .dep, ontology, rem*.csv)
for f in os.listdir(f'{INPUT_BASE}/data'):
    shutil.copy(f'{INPUT_BASE}/data/{f}', f'{WORK}/data/{f}')

# cooc_matrix_final2.csv sits at the top of the bundle (alongside src/ and data/)
shutil.copy(f'{INPUT_BASE}/cooc_matrix_final2.csv', f'{WORK}/cooc_matrix_final2.csv')

# test_ontology_keys.csv is under src/
shutil.copy(f'{INPUT_BASE}/src/test_ontology_keys.csv', f'{WORK}/test_ontology_keys.csv')

os.chdir(WORK)
print('cwd:', os.getcwd())
print('working/ contents:', sorted(os.listdir(WORK)))

# Verify the .const files made it
for year in ['2015', '2016']:
    for split in ['train', 'test']:
        p = f'data/{split}{year}restaurant.txt.const'
        assert os.path.exists(p), f'MISSING: {p}'
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f'  {p}: {n} lines')

# Also confirm cooc + ontology landed
assert os.path.exists('cooc_matrix_final2.csv'), 'cooc matrix missing'
assert os.path.exists('test_ontology_keys.csv'), 'ontology keys missing'
print('cooc + ontology OK')

In [ ]:
!pip install pytorch_transformers==1.2.0 --quiet

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

import pytorch_transformers
print('pytorch_transformers:', pytorch_transformers.__version__)

In [ ]:
import nbformat

NB_PATH = f'{INPUT_BASE}/src/7GCN.ipynb'
nb = nbformat.read(NB_PATH, as_version=4)

# Definition cells run up through cell 25 (get_args), inclusive.
DEFINITION_CELLS = list(range(0, 25))

for i, cell in enumerate(nb.cells):
    if i not in DEFINITION_CELLS:
        continue
    if cell.cell_type != 'code':
        continue
    src = ''.join(cell.source)
    if not src.strip():
        continue
    try:
        exec(src, globals())
        print(f"✓ cell {i:2d} OK ({len(src)} chars)")
    except Exception as e:
        print(f"✗ cell {i:2d} FAILED: {type(e).__name__}: {e}")
        raise

In [ ]:
import pandas as pd

print('Loading co-occurrence matrix...')
cooc = pd.read_csv('cooc_matrix_final2.csv', index_col=0)
print(f'  cooc shape: {cooc.shape}')

print('Loading ontology words...')
onto_words_df = pd.read_csv('test_ontology_keys.csv', sep=';')
onto_words = [item for sublist in onto_words_df.values.tolist() for item in sublist]
onto_words = list(dict.fromkeys(onto_words))
print(f'  ontology words count: {len(onto_words)}')

In [ ]:
import os, json, time, gc
#Below are the top-3 configurations from the TPE search for 4GCN. Select and copy to CANDIDATES un-commented to run.
#CANDIDATES = [
 #   {'tag': 'rank1', 'learning_rate': 8.836537235645673e-06, 'dropout': 0.25, 'l2reg': 0.0005059218096160434},
  #  {'tag': 'rank3', 'learning_rate': 1.933e-05,             'dropout': 0.35, 'l2reg': 9.251e-04},
   # {'tag': 'rank4', 'learning_rate': 1.020e-05,             'dropout': 0.20, 'l2reg': 1.152e-03},
#]

#winner below from 2015 => cashing in these hyperparamn for 2016

CANDIDATES = [
      {'tag': 'rank1', 'learning_rate': 8.836537235645673e-06, 'dropout': 0.25, 'l2reg': 0.0005059218096160434},
  ]

CONCAT_DROPOUT_FIXED = 0.2285714285714286
SEED   = 7
YEAR   = '2016'
EPOCHS = 15
OUTDIR = '/kaggle/working/val15_2015'
os.makedirs(OUTDIR, exist_ok=True)

for cand in CANDIDATES:
    tag = cand['tag']
    out_json = f'{OUTDIR}/val15_{YEAR}_seed{SEED}_{tag}.json'

    if os.path.exists(out_json):
        prev = json.load(open(out_json))
        print(f'[skip] {tag} already done: val={prev["max_val_acc"]:.4f} test={prev["test_acc"]:.4f}')
        continue

    print(f'\n{"="*70}\nVALIDATING {tag}  lr={cand["learning_rate"]:.3e} '
          f'drop={cand["dropout"]} l2={cand["l2reg"]:.3e}  (15 epochs, {YEAR}, seed {SEED})\n{"="*70}')

    global opt
    opt = get_args(
        model_type='tri_gcn', constgcn=False,
        tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
        year=YEAR, seed=SEED,
        learning_rate=float(cand['learning_rate']),
        dropout=float(cand['dropout']),
        concat_dropout=CONCAT_DROPOUT_FIXED,
        l2reg=float(cand['l2reg']),
        num_epoch=EPOCHS, batch_size=4,
        save_models='none', fusion_type='concat', use_ensemble=False,
        cooc=cooc, onto_words=onto_words,
        do_train=True, do_eval=True,
        path=f'{OUTDIR}/tmp_{tag}',
    )
    opt.device = torch.device('cuda')

    t0 = time.time()
    max_val_acc, test_acc, test_f1 = main(opt)
    elapsed = time.time() - t0

    result = {
        'tag': tag, 'max_val_acc': float(max_val_acc),
        'test_acc': float(test_acc), 'test_f1': float(test_f1),
        'elapsed_sec': round(elapsed, 1),
        'seed': SEED, 'year': YEAR, 'epochs': EPOCHS,
        'learning_rate': float(cand['learning_rate']),
        'dropout': float(cand['dropout']),
        'concat_dropout': CONCAT_DROPOUT_FIXED,
        'l2reg': float(cand['l2reg']),
    }
    with open(out_json, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'[done] {tag}: val={max_val_acc:.4f} test={test_acc:.4f} '
          f'f1={test_f1:.4f} ({elapsed/60:.1f} min) -> {out_json}')

    gc.collect(); torch.cuda.empty_cache()

print(f'\n{"="*70}\nVALIDATION SUMMARY (15 epochs, {YEAR}, seed {SEED})\n{"="*70}')
rows = []
for cand in CANDIDATES:
    p = f'{OUTDIR}/val15_{YEAR}_seed{SEED}_{cand["tag"]}.json'
    if os.path.exists(p):
        rows.append(json.load(open(p)))
rows.sort(key=lambda r: r['max_val_acc'], reverse=True)
print(f"{'tag':>6} {'val_acc':>8} {'test_acc':>9} {'lr':>11} {'drop':>5} {'l2reg':>10}")
for r in rows:
    print(f"{r['tag']:>6} {r['max_val_acc']:>8.4f} {r['test_acc']:>9.4f} "
          f"{r['learning_rate']:>11.3e} {r['dropout']:>5.2f} {r['l2reg']:>10.3e}")
if rows:
    w = rows[0]
    print(f'\nWINNER on {YEAR}: {w["tag"]}  '
          f'lr={w["learning_rate"]:.3e} drop={w["dropout"]} l2={w["l2reg"]:.3e}')
    print(f'  val={w["max_val_acc"]:.4f} test={w["test_acc"]:.4f}')
    print(f'\nNEXT SESSION: run this winning config on 2016 (change YEAR=2016).')